# Langkah 8b — Pelabelan Manual Sisa Ulasan (kuota API habis)

Kuota API hanya cukup untuk **7.048 dari 8.641** ulasan. Sisanya (~1.593, hampir semua
**Semarang**) dilabeli **manual** lewat Excel, dengan format **sama persis** dengan output
model (dimension + polarity + **sub_issue** + **quote**) supaya bisa digabung langsung ke
`findings_full.csv`.

> `sub_issue` & `quote` = jantung fitur dashboard ("sumber komplain" + "lihat ulasan asli"),
> jadi keduanya WAJIB diisi, bukan hanya kategori.

Dua fase:
- **Fase A** (sekarang): tulis template Excel, dibagi 3 anotator.
- **Fase B** (setelah selesai): gabungkan label manual + output model → `findings_full.csv` final.

In [1]:
import sys, os, json
from pathlib import Path

ROOT = Path('..').resolve()
DATA_DIR = ROOT.parent / 'data'
OUT = ROOT / 'outputs'
sys.path.insert(0, str(ROOT))

import importlib
import pandas as pd
import absa.goldset
importlib.reload(absa.goldset)
from absa.goldset import write_label_template, load_label_template

df = pd.read_csv(DATA_DIR / 'reviews_cleaned_rating_1_2.csv')
df['review_id'] = df['review_id'].astype(str)
print(f'Total ulasan: {len(df)}')

Total ulasan: 8641


## Fase A.1 — Identifikasi ulasan yang BELUM diproses model

In [2]:
# Ulasan yang sudah ada di output model
recs = [json.loads(l) for l in open(OUT / 'full_raw.jsonl', encoding='utf-8')]
done_ids = {r['review_id'] for r in recs}

todo = df[~df['review_id'].isin(done_ids)].copy()
print(f'Sudah diproses model : {len(done_ids)} ulasan')
print(f'Perlu label manual   : {len(todo)} ulasan')
print(f'\nSebaran wilayah (todo): {todo["wilayah"].value_counts().to_dict()}')
print(f'Sebaran rating (todo) : {todo["rating"].value_counts().to_dict()}')

Sudah diproses model : 7048 ulasan
Perlu label manual   : 1593 ulasan

Sebaran wilayah (todo): {'Semarang': 1589, 'Surabaya': 4}
Sebaran rating (todo) : {1: 1309, 2: 284}


## Fase A.2 — Tulis template (dibagi 3 anotator)

Tiap anotator mendapat ~1/3 ulasan. Format: **satu baris per temuan**. Tiap ulasan punya
3 baris kosong; kalau temuan lebih banyak, tambahkan baris (salin `review_id` yang sama).
Kalau ulasan tidak berisi keluhan/pujian (off-topic/pertanyaan murni), biarkan keempat kolom
temuan kosong.

In [3]:
# Bagi todo jadi 3 bagian (urutan acak tapi reproducible)
todo_shuffled = todo.sample(frac=1, random_state=42).reset_index(drop=True)
splits = [todo_shuffled.iloc[i::3] for i in range(3)]   # selang-seling, ukuran seimbang

for i, part in enumerate(splits, start=1):
    path = OUT / f'manual_label_{i}.xlsx'
    write_label_template(part, path, blank_rows_per_review=3)
    print(f'  manual_label_{i}.xlsx : {len(part)} ulasan (untuk anotator {i})')

print('\n>> Bagikan ke 3 anotator. Lanjut Fase B setelah semua terisi.')

  manual_label_1.xlsx : 531 ulasan (untuk anotator 1)
  manual_label_2.xlsx : 531 ulasan (untuk anotator 2)
  manual_label_3.xlsx : 531 ulasan (untuk anotator 3)

>> Bagikan ke 3 anotator. Lanjut Fase B setelah semua terisi.


## Panduan Pengisian (WAJIB dibaca anotator)

Untuk **setiap ulasan**, baca teksnya lalu tulis **satu baris per temuan**:

| Kolom | Isi |
|---|---|
| **dimension** | Salah satu: `Responsiveness`, `Reliability`, `Assurance`, `Empathy`, `Tangibles`, `Umum` |
| **polarity** | `neg` (keluhan) atau `pos` (pujian) |
| **sub_issue** | Frasa pendek 2–5 kata, mis. `antre lama`, `petugas judes`, `obat kosong`. Konsisten! |
| **quote** | Kutipan **persis** dari teks ulasan (salin apa adanya, jangan diperbaiki) |

**Aturan:**
1. Satu ulasan boleh punya beberapa temuan → beberapa baris (salin `review_id` yang sama).
2. Satu keluhan spesifik = satu baris. Jangan pecah jadi dua.
3. **Umum** hanya jika TIDAK ada aspek spesifik ("pelayanan buruk" tanpa rincian). Kalau ada
   aspek spesifik, pakai dimensi itu, **jangan** Umum.
4. Pertanyaan murni tanpa sentimen ("Apa bisa pakai BPJS?") atau basa-basi (doa/salam) →
   **biarkan semua kolom temuan kosong** (cukup `review_id` terisi).
5. Definisi dimensi & batas-batasnya sama dengan panduan gold set (Langkah 4).

---
# Fase B — Gabungkan label manual ke findings_full.csv

Jalankan setelah `manual_label_1/2/3.xlsx` terisi.

In [3]:
# 1) Muat output MODEL → format temuan
def model_findings_long(records):
    rows = []
    for rec in records:
        fs = rec.get('findings', [])
        if isinstance(fs, str):
            try: fs = json.loads(fs)
            except json.JSONDecodeError: continue
        for f in fs:
            if not isinstance(f, dict): continue
            rows.append({
                'review_id': rec['review_id'],
                'puskesmas_id': rec.get('puskesmas_id'),
                'puskesmas_name': rec.get('puskesmas_name'),
                'wilayah': rec.get('wilayah'),
                'rating': rec.get('rating'),
                'dimension': f.get('dimension'),
                'polarity': f.get('polarity'),
                'sub_issue': f.get('sub_issue'),
                'quote': f.get('quote'),
                'sumber': 'model',
            })
    return pd.DataFrame(rows)

model_df = model_findings_long(recs)
print(f'Temuan dari model: {len(model_df)} (dari {model_df["review_id"].nunique()} ulasan)')

Temuan dari model: 15198 (dari 6692 ulasan)


In [4]:
# 2) Muat label MANUAL → format yang sama, lengkapi metadata puskesmas dari df
meta = df.set_index('review_id')[['puskesmas_id','puskesmas_name','wilayah','rating']]

manual_parts = []
for i in (1, 2, 3):
    p = OUT / f'manual_label_{i}.xlsx'
    if p.exists():
        lab = load_label_template(p)
        manual_parts.append(lab)
        print(f'  manual_label_{i}.xlsx : {len(lab)} temuan')
    else:
        print(f'  manual_label_{i}.xlsx : (belum ada)')

manual_df = pd.concat(manual_parts, ignore_index=True) if manual_parts else pd.DataFrame()
if len(manual_df):
    manual_df = manual_df.join(meta, on='review_id')
    manual_df['sumber'] = 'manual'
    print(f'\nTotal temuan manual: {len(manual_df)} (dari {manual_df["review_id"].nunique()} ulasan)')

  manual_label_1.xlsx : 731 temuan
  manual_label_2.xlsx : 749 temuan
  manual_label_3.xlsx : 697 temuan

Total temuan manual: 2177 (dari 1575 ulasan)


In [5]:
# 3) Gabungkan, validasi, simpan
combined = pd.concat([model_df, manual_df], ignore_index=True)

# validasi: dimensi & polaritas valid
from absa.prompts import CATEGORIES
bad_dim = ~combined['dimension'].isin(CATEGORIES)
bad_pol = ~combined['polarity'].isin(['neg','pos'])
assert not bad_dim.any(), f'Dimensi tak valid: {combined.loc[bad_dim,"dimension"].unique()}'
assert not bad_pol.any(), f'Polaritas tak valid: {combined.loc[bad_pol,"polarity"].unique()}'

# cakupan ulasan
covered = set(combined['review_id'])
all_ids = set(df['review_id'])
uncovered = all_ids - covered
print(f'Total temuan gabungan: {len(combined)}')
print(f'  - model : {(combined["sumber"]=="model").sum()}')
print(f'  - manual: {(combined["sumber"]=="manual").sum()}')
print(f'Ulasan tercakup (>=1 temuan): {len(covered & all_ids)} / {len(all_ids)}')
print(f'Ulasan tanpa temuan        : {len(uncovered)} (off-topic/inquiry/tidak relevan)')

combined.to_csv(OUT / 'findings_full.csv', index=False, encoding='utf-8-sig')
print('\nTersimpan: outputs/findings_full.csv  → siap untuk Langkah 9 (statistik).')

Total temuan gabungan: 17375
  - model : 15198
  - manual: 2177
Ulasan tercakup (>=1 temuan): 8267 / 8641
Ulasan tanpa temuan        : 374 (off-topic/inquiry/tidak relevan)

Tersimpan: outputs/findings_full.csv  → siap untuk Langkah 9 (statistik).


## Catatan metodologis (untuk laporan)

Sumber label dilacak di kolom **`sumber`** (`model` / `manual`). Karena sisa manual
hampir seluruhnya **Semarang**, di Langkah 9 lakukan **sensitivity check**: bandingkan
distribusi dimensi Semarang (manual) vs wilayah lain (model) untuk memastikan tidak ada
pergeseran sistematis akibat perbedaan sumber pelabelan.